# Attention Is All You Need — Causal Multi-Head Attention & KV Cache

A hands-on, step-by-step walkthrough using a tiny example (5 tokens, `d_model = 8`, 2 heads) so every tensor is small enough to read.

**Roadmap**
1. Toy input: one embedding vector per token
2. Naive attention: raw similarity `x @ xᵀ`
3. Learned projections: Query / Key / Value
4. Scores, scaling, causal mask, softmax
5. Full multi-head attention (and a check against PyTorch's built-in)
6. **KV cache**: why we need it, how it works, and a proof it gives identical results

> **Shape notation:** `B` = batch, `S` = sequence length, `D` = `d_model`, `H` = number of heads, `d_k` = `D / H` (size of each head).

## 0. Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Fixed seed so every run produces the same "random" embeddings and weights
torch.manual_seed(0)

# Print tensors compactly: 2 decimals, no scientific notation
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch", torch.__version__)

torch 2.14.1+cpu


In [2]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Pretty-print a 2-D tensor with token names on the rows and columns.

    Rows    = the token doing the "looking" (query side).
    Columns = the token being "looked at"   (key side).
    Note: defaults to the global TOKENS list, which is defined in the next section.
    """
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    # Header line with column labels
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    # One line per row: label + formatted numbers
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

## 1. Toy input

A real model would turn text into token ids, then look each id up in an embedding table. Here we skip that and just use **random vectors** as stand-in embeddings. Each token is a row of `d_model = 8` numbers.

In [3]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

batch   = 1                  # number of sequences processed at once
seq_len = len(TOKENS)        # number of tokens (S)
d_model = 8                  # width of each token vector (D)
n_heads = 2                  # attention heads (H)
d_k     = d_model // n_heads # width of each head; d_model must divide evenly by n_heads

# Mock embeddings: shape (batch, seq_len, d_model)
x = torch.randn(batch, seq_len, d_model)

print("tokens :", TOKENS)
print("x.shape:", tuple(x.shape), " (batch, time, width)")
print()
print("x[0] -- one row per token:")
print(x)

tokens : ['the', 'cat', 'sat', 'on', 'mat']
x.shape: (1, 5, 8)  (batch, time, width)

x[0] -- one row per token:
tensor([[[-1.13, -1.15, -0.25, -0.43,  0.85,  0.69, -0.32, -2.12],
         [ 0.32, -1.26,  0.35,  0.31,  0.12,  1.24,  1.12, -0.25],
         [-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85],
         [-0.22, -0.74,  0.56,  0.26, -0.17, -0.68,  0.94,  0.49],
         [ 1.20,  0.08, -1.20, -0.00, -0.52, -0.31, -1.58,  1.71]]])


In [4]:
# Look at just the first 5 of the 8 dimensions to keep the printout small
col_labels = [f"d{i}" for i in range(5)]
show(x[0][:, :5], "x, first 5 columns", col_labels=col_labels)

x, first 5 columns
             d0      d1      d2      d3      d4
   the    -1.13   -1.15   -0.25   -0.43    0.85
   cat     0.32   -1.26    0.35    0.31    0.12
   sat    -1.35   -1.70    0.57    0.79    0.60
    on    -0.22   -0.74    0.56    0.26   -0.17
   mat     1.20    0.08   -1.20   -0.00   -0.52



## 2. Naive attention: "how similar are two tokens?"

The simplest idea: a **dot product** between two vectors is large when they point in the same direction. Multiplying `x` by its own transpose gives the dot product of *every token with every token* in one matrix multiply.

`(S, D) @ (D, S) -> (S, S)`

Problems with this naive version (which the next steps fix):
- The diagonal is always large (a token is most similar to itself) — not very informative.
- There are **no learnable parameters**, so the model can't learn *what* to look for.
- It lets every token see **future** tokens, which is cheating for next-word prediction.

In [5]:
naive_scores = x[0] @ x[0].T   # (5, 8) @ (8, 5) -> (5, 5)

print("x[0].shape:", tuple(x[0].shape))
print("naive_scores.shape:", tuple(naive_scores.shape))

# Notice the large values on the diagonal: each token matches itself best
show(naive_scores, "naive_scores[i][j] = how much token i matches token j")

x[0].shape: (5, 8)
naive_scores.shape: (5, 5)
naive_scores[i][j] = how much token i matches token j
            the     cat     sat      on     mat
   the     8.62    2.00   -1.39   -1.10   -4.91
   cat     2.00    4.77   -0.54    1.21   -2.77
   sat    -1.39   -0.54   11.98    3.61    1.41
    on    -1.10    1.21    3.61    2.59   -1.35
   mat    -4.91   -2.77    1.41   -1.35    8.67



## 3. Learned projections: Query, Key, Value

Instead of comparing raw embeddings, we first project each token into three different "roles" using learned weight matrices:

| Role | Question it answers |
|------|---------------------|
| **Query (Q)** | "What am I looking for?" |
| **Key (K)** | "What do I contain / advertise?" |
| **Value (V)** | "What information do I hand over if someone attends to me?" |

Attention score between token *i* and token *j* = `Q[i] · K[j]`.
The final output is a weighted mix of the **Values**, weighted by those (softmaxed) scores.

We also build the **causal mask** here: a lower-triangular matrix of `True/False`. Row *i* = the token asking, column *j* = the token being looked at. `True` means "allowed to look", so token *i* may only look at tokens `j <= i` (itself and the past).

In [6]:
# Three learned projections, each maps d_model -> d_model.
# (Each one will later be split into n_heads chunks of size d_k.)
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

# Output projection: mixes the concatenated heads back together
W_o = nn.Linear(d_model, d_model)

# Project every token into its query / key / value vector (shape unchanged: B, S, D)
q = W_q(x)
k = W_k(x)
v = W_v(x)

# Causal mask: lower-triangular, True = "may attend"
mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))

print("x.shape:", tuple(x.shape))
print("q.shape:", tuple(q.shape))
print("k.shape:", tuple(k.shape))
print("v.shape:", tuple(v.shape))
print()
print("causal mask (True = allowed to attend):")
print(mask.int())  # printed as 1/0 for readability

x.shape: (1, 5, 8)
q.shape: (1, 5, 8)
k.shape: (1, 5, 8)
v.shape: (1, 5, 8)

causal mask (True = allowed to attend):
tensor([[1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]], dtype=torch.int32)


In [7]:
# Same token ('sat', index 2) seen through three different lenses.
# The projections turn one embedding into different vectors for different roles.
print("x[sat]:", x[0][2])
print("q[sat]:", q[0][2])
print("k[sat]:", k[0][2])

x[sat]: tensor([-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85])
q[sat]: tensor([ 0.24, -0.66, -0.16, -0.36,  0.03, -0.97,  0.44, -0.29], grad_fn=<SelectBackward0>)
k[sat]: tensor([-0.35,  0.41, -0.05,  0.72, -0.07,  0.05,  1.05,  0.49], grad_fn=<SelectBackward0>)


## 4. Scores: `Q @ Kᵀ`

Multiplying queries by transposed keys gives a `(S, S)` table: row *i* = how strongly token *i*'s query matches each token's key.

Two more steps happen in the real layer (shown in the next section):
1. **Scale** by `√d_k`. Without it, dot products grow with the vector size, pushing softmax into near one-hot outputs with tiny gradients.
2. **Mask + softmax.** Set forbidden (future) positions to `-inf` so softmax gives them exactly 0 weight, then each row sums to 1.

Below are the *raw* scores (no scaling, no mask, single head) so you can see the starting point.

In [8]:
# (B, S, D) @ (B, D, S) -> (B, S, S)
# transpose(-2, -1) swaps the last two dims, so it works with any number of batch dims
scores = q @ k.transpose(-2, -1)

print("scores.shape:", tuple(scores.shape))
print()
show(scores[0], "scores = q @ k^T  (raw: unscaled, unmasked)")

scores.shape: (1, 5, 5)

scores = q @ k^T  (raw: unscaled, unmasked)
            the     cat     sat      on     mat
   the     0.70   -1.33    0.45   -0.22    0.33
   cat    -0.69   -0.17   -1.03   -0.20    0.78
   sat    -0.51   -0.31   -0.34   -0.35    0.07
    on    -0.45    0.47   -0.55   -0.03   -0.27
   mat    -0.87   -0.58   -0.46   -0.53    1.74



## 5. Full multi-head causal attention

**Why multiple heads?** Each head gets its own slice (`d_k` dims) of Q, K and V and computes its own attention pattern. One head might track "previous word", another "subject of the sentence", etc. Their outputs are concatenated and mixed by `W_o`.

**Reshaping trick** (`B, S, D` → `B, H, S, d_k`):
```
.view(B, S, H, d_k)     # split D into H chunks of d_k
.transpose(1, 2)        # move heads next to batch so each head is processed independently
```

**Pipeline**
```
x -> Q,K,V -> split heads -> scores = QKᵀ/√d_k -> causal mask -> softmax -> weights @ V
  -> merge heads -> W_o -> output
```

This cell is self-contained (it recreates `x` and the weights), so its random numbers differ from the earlier sections.

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

TOKENS = ["the", "cat", "sat", "on", "mat"]

batch = 1
seq_len = len(TOKENS)
d_model = 8
n_heads = 2
d_k = d_model // n_heads

# Mock input embeddings: (batch, seq_len, d_model)
x = torch.randn(batch, seq_len, d_model)

# Projection layers
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)
W_o = nn.Linear(d_model, d_model)


def multiheadattention(x, W_q, W_k, W_v, W_o, mask, n_heads):
    """Causal multi-head self-attention (no cache).

    x    : (B, S, D) input embeddings
    mask : (>=S, >=S) bool lower-triangular matrix, True = allowed to attend
    Returns (output (B, S, D), attention weights (B, H, S, S)).
    """
    batch, seq_len, d_model = x.shape
    d_k = d_model // n_heads

    # 1) Project, then split D into H heads: (B, S, D) -> (B, H, S, d_k)
    q = W_q(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)
    v = W_v(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)

    # 2) Similarity of every query with every key, per head: (B, H, S, S)
    scores = q @ k.transpose(-2, -1)
    scores = scores / (d_k ** 0.5)           # scale to keep softmax well-behaved

    # 3) Causal mask: forbidden positions -> -inf -> softmax weight of exactly 0
    causal = mask[:seq_len, :seq_len].to(x.device)
    scores = scores.masked_fill(~causal, float("-inf"))

    # 4) Turn scores into probabilities over the keys (each row sums to 1)
    weights = F.softmax(scores, dim=-1)

    # 5) Weighted sum of values: (B, H, S, S) @ (B, H, S, d_k) -> (B, H, S, d_k)
    out = weights @ v

    # 6) Merge heads back: (B, H, S, d_k) -> (B, S, H, d_k) -> (B, S, D)
    #    .contiguous() is required before .view() after a transpose
    out = out.transpose(1, 2).contiguous().view(batch, seq_len, d_model)

    # 7) Final output projection
    return W_o(out), weights


# Causal mask
mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))

# Forward pass
out, weights = multiheadattention(x, W_q, W_k, W_v, W_o, mask, n_heads)

print("Tokens:", TOKENS)
print("Input shape:", x.shape)
print("Q/K/V shape (per head):", (batch, n_heads, seq_len, d_k))
print("Attention weights shape:", weights.shape)
print("Output shape:", out.shape)

# Display attention weights for each head.
# Row = token asking, column = token being attended to.
# Upper triangle is 0 (future is masked); every row sums to 1.
for h in range(n_heads):
    print(f"\nHead {h + 1} attention:")
    print("       " + "  ".join(f"{t:>5}" for t in TOKENS))
    for i, token in enumerate(TOKENS):
        row = "  ".join(f"{v:.3f}" for v in weights[0, h, i].tolist())
        print(f"{token:>5}: {row}")

Tokens: ['the', 'cat', 'sat', 'on', 'mat']
Input shape: torch.Size([1, 5, 8])
Q/K/V shape (per head): (1, 2, 5, 4)
Attention weights shape: torch.Size([1, 2, 5, 5])
Output shape: torch.Size([1, 5, 8])

Head 1 attention:
         the    cat    sat     on    mat
  the: 1.000  0.000  0.000  0.000  0.000
  cat: 0.617  0.383  0.000  0.000  0.000
  sat: 0.714  0.124  0.162  0.000  0.000
   on: 0.202  0.221  0.319  0.257  0.000
  mat: 0.326  0.141  0.105  0.218  0.210

Head 2 attention:
         the    cat    sat     on    mat
  the: 1.000  0.000  0.000  0.000  0.000
  cat: 0.683  0.317  0.000  0.000  0.000
  sat: 0.227  0.161  0.613  0.000  0.000
   on: 0.312  0.196  0.236  0.255  0.000
  mat: 0.131  0.260  0.304  0.146  0.158


### Sanity check against PyTorch

`F.scaled_dot_product_attention` is PyTorch's built-in fused attention. With `is_causal=True` it should reproduce our manual result exactly (up to floating-point error).

In [10]:
# Trust-but-verify: the from-scratch implementation must reproduce PyTorch's
# fused kernel. Dropout plays no role in either path, so a match is not luck;
# atol=1e-6 only absorbs float-ordering differences.
with torch.no_grad():
    # Rebuild per-head q, k, v: (B, H, S, d_k)
    q_h = W_q(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)
    k_h = W_k(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)
    v_h = W_v(x).view(batch, seq_len, n_heads, d_k).transpose(1, 2)

    ref = F.scaled_dot_product_attention(q_h, k_h, v_h, is_causal=True)
    ref = W_o(ref.transpose(1, 2).contiguous().view(batch, seq_len, d_model))

print("matches PyTorch built-in:", torch.allclose(out, ref, atol=1e-6))

matches PyTorch built-in: True


# Introducing the KV Cache

## The problem

When a language model **generates** text, it produces one token at a time. Without any optimization, each new token means re-running attention over the *entire* prefix, so most of the work is repeated.

## The key observation

Because of the **causal mask**, a token only depends on itself and earlier tokens. So when we append a new token:
- the **keys and values of old tokens do not change**,
- the **outputs of old tokens do not change**.

Only the new token needs fresh computation. So we **store (cache) the K and V** of every past token and, at each step, compute just the new token's Q, K, V.

(We cache K and V but not Q: old queries are never needed again, since old outputs are already finished.)

### Demo: old outputs never change

Run attention on growing prefixes `the` → `the cat` → `the cat sat` → ... and watch the output for the first token `the`. It is identical every time, while the newest token's output is new each step.

In [11]:
def row(t, style="{:6.2f}"):
    """Format a 1-D tensor as a compact one-line string."""
    return "[" + "  ".join(style.format(v) for v in t.tolist()) + "]"

with torch.no_grad():
    for step in range(1, seq_len + 1):
        # Feed only the first `step` tokens (the prefix so far)
        mha_out, _ = multiheadattention(x[:, :step, :], W_q, W_k, W_v, W_o, mask, n_heads)

        prefix = " ".join(TOKENS[:step])
        print(f"step: {step}")
        print(f"Prefix Input: {prefix}")
        # [0, 0, :] = batch 0, token position 0 ('the'), all features
        print("Out['the']  :", row(mha_out[0, 0, :]))    # constant across steps
        # [0, -1, :] = the newest token in the prefix
        print("Out[newest] :", row(mha_out[0, -1, :]))   # changes each step
        print()

step: 1
Prefix Input: the
Out['the']  : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]
Out[newest] : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]

step: 2
Prefix Input: the cat
Out['the']  : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]
Out[newest] : [  0.01    0.09   -0.03   -0.01    0.02   -0.27   -0.40    0.21]

step: 3
Prefix Input: the cat sat
Out['the']  : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]
Out[newest] : [ -0.39   -0.04    0.23   -0.21   -0.00   -0.80    0.06    0.14]

step: 4
Prefix Input: the cat sat on
Out['the']  : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]
Out[newest] : [ -0.13    0.12    0.07   -0.18    0.11   -0.48   -0.23    0.27]

step: 5
Prefix Input: the cat sat on mat
Out['the']  : [ -0.06   -0.14   -0.12   -0.16   -0.14   -0.19   -0.49    0.10]
Out[newest] : [ -0.20   -0.08    0.09   -0.22   -0.11   -0.51   -0.09    0.24]



## How the cache works, step by step

Suppose `the cat sat` is already processed and we now feed the new token `on`:

| Tensor | Shape | Where it comes from |
|--------|-------|---------------------|
| `q` | `(B, H, 1, d_k)` | only the **new** token |
| `k`, `v` | `(B, H, 3+1, d_k)` | 3 from the cache **+** 1 new, joined with `torch.cat(..., dim=2)` |

`dim=2` is the sequence/time axis of a `(B, H, S, d_k)` tensor.

In [12]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

past_length = 3
new_length = 1

past = x[:, :3, :]   # tokens already processed: "the", "cat", "sat"
new  = x[:, 3:4, :]  # the token fed in this step: "on"  (3:4 keeps the time dim)

# --- Build the cache from the past tokens (in real use this was filled by earlier steps)
past_k = W_k(past).view(batch, past_length, n_heads, d_k).transpose(1, 2)
past_v = W_v(past).view(batch, past_length, n_heads, d_k).transpose(1, 2)
kv_cache = (past_k, past_v)

# --- Compute q, k, v for the NEW token only
q = W_q(new).view(batch, new_length, n_heads, d_k).transpose(1, 2)
k = W_k(new).view(batch, new_length, n_heads, d_k).transpose(1, 2)
v = W_v(new).view(batch, new_length, n_heads, d_k).transpose(1, 2)

# --- Read the cache and append the new k, v along the time axis
past_k, past_v = kv_cache
past_length = past_k.size(2)     # number of cached tokens
seq_length = new.shape[1]        # number of new tokens

k = torch.cat([past_k, k], dim=2)
v = torch.cat([past_v, v], dim=2)

total_length = past_length + seq_length   # how many tokens attention can see


print("past_len :", past_length,  " tokens already in the cache")
print("seq_len  :", seq_length,   " tokens fed in this step")
print("total_len:", total_length, " tokens attention can see")
print()
print("q:", tuple(q.shape), " <- 1 query   (q for 'the','cat','sat' is NOT rebuilt)")
print("k:", tuple(k.shape), " <- 4 keys    (3 reused from cache + 1 new)")
print("v:", tuple(v.shape), " <- 4 values  (3 reused from cache + 1 new)")

past_len : 3  tokens already in the cache
seq_len  : 1  tokens fed in this step
total_len: 4  tokens attention can see

q: (1, 2, 1, 4)  <- 1 query   (q for 'the','cat','sat' is NOT rebuilt)
k: (1, 2, 4, 4)  <- 4 keys    (3 reused from cache + 1 new)
v: (1, 2, 4, 4)  <- 4 values  (3 reused from cache + 1 new)


### Finishing the step: the mask with a cache

The scores now have shape `(B, H, 1, 4)`: one query against four keys. The mask must match, so instead of the full `S x S` mask we take only the rows for the **new** queries:

```python
causal = mask[past_len : total_len, : total_len]   # shape (new_len, total_len)
```

- `past_len : total_len` selects the rows of the new tokens (skips the old ones).
- `: total_len` selects the columns of all visible tokens.

Here the new token is row 3, which may see columns 0-3, so every entry is `True`. Then we verify the result equals the **last row** of a full, cache-free pass.

In [13]:
with torch.no_grad():
    # Scores: 1 query vs 4 keys -> (B, H, 1, 4)
    scores = q @ k.transpose(-2, -1) / (d_k ** 0.5)

    # Mask rows for the new token only
    causal = mask[past_length:total_length, :total_length]
    print("mask slice for the new token:", causal.int().tolist())

    scores  = scores.masked_fill(~causal, float("-inf"))
    weights = F.softmax(scores, dim=-1)                      # (B, H, 1, 4)
    step_out = weights @ v                                   # (B, H, 1, d_k)
    step_out = step_out.transpose(1, 2).contiguous().view(batch, new_length, d_model)
    step_out = W_o(step_out)                                 # (B, 1, D)

    # Reference: full recompute over "the cat sat on", take the last token
    full_out, _ = multiheadattention(x[:, :4, :], W_q, W_k, W_v, W_o, mask, n_heads)

print("cached output  :", row(step_out[0, 0]))
print("full recompute :", row(full_out[0, -1]))
print("identical      :", torch.allclose(step_out[:, 0], full_out[:, -1], atol=1e-6))

mask slice for the new token: [[1, 1, 1, 1]]
cached output  : [ -0.13    0.12    0.07   -0.18    0.11   -0.48   -0.23    0.27]
full recompute : [ -0.13    0.12    0.07   -0.18    0.11   -0.48   -0.23    0.27]
identical      : True


## The complete cached attention function

This mirrors the `MultiHeadAttention.forward(x, kv_cache, use_cache)` method in `attention.py`:

1. Project the **new** tokens into q, k, v.
2. If a cache exists, `cat` the old K/V in front of the new ones.
3. Return `(k, v)` as the updated cache for the next step.
4. Slice the mask with `[past_len:total_len, :total_len]`.

Then we generate token by token (feeding one token at a time) and check that the stacked outputs equal one full pass over the whole sequence.

In [14]:
def multiheadattention_cached(x, W_q, W_k, W_v, W_o, mask, n_heads, kv_cache=None):
    """Causal multi-head attention with an optional KV cache.

    x        : (B, new_len, D)  ONLY the new tokens
    kv_cache : None, or (past_k, past_v), each (B, H, past_len, d_k)
    Returns  : (output (B, new_len, D), updated cache (k, v))
    """
    batch, new_len, d_model = x.shape
    d_k = d_model // n_heads

    # q, k, v for the new tokens only: (B, H, new_len, d_k)
    q = W_q(x).view(batch, new_len, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(batch, new_len, n_heads, d_k).transpose(1, 2)
    v = W_v(x).view(batch, new_len, n_heads, d_k).transpose(1, 2)

    past_len = 0
    if kv_cache is not None:
        past_k, past_v = kv_cache
        past_len = past_k.size(2)
        # Old keys/values go first so positions stay in chronological order
        k = torch.cat([past_k, k], dim=2)
        v = torch.cat([past_v, v], dim=2)

    total_len = past_len + new_len
    updated_cache = (k, v)   # grows by new_len every call

    # (B, H, new_len, d_k) @ (B, H, d_k, total_len) -> (B, H, new_len, total_len)
    scores = (q @ k.transpose(-2, -1)) / (d_k ** 0.5)

    # Rows = new queries only, columns = every visible key
    causal = mask[past_len:total_len, :total_len]
    scores = scores.masked_fill(~causal, float("-inf"))

    weights = F.softmax(scores, dim=-1)
    out = (weights @ v).transpose(1, 2).contiguous().view(batch, new_len, d_model)
    return W_o(out), updated_cache


with torch.no_grad():
    cache = None
    step_outputs = []
    for t in range(seq_len):
        new_token = x[:, t:t + 1, :]   # feed ONE token per step
        out_t, cache = multiheadattention_cached(
            new_token, W_q, W_k, W_v, W_o, mask, n_heads, kv_cache=cache
        )
        step_outputs.append(out_t)
        print(f"step {t + 1}: fed '{TOKENS[t]}' | cache now holds {cache[0].size(2)} keys & values")

    incremental = torch.cat(step_outputs, dim=1)          # (B, S, D)
    full, _ = multiheadattention(x, W_q, W_k, W_v, W_o, mask, n_heads)

print()
print("incremental (cached) == full pass:", torch.allclose(incremental, full, atol=1e-6))
cache_bytes = sum(t.numel() * t.element_size() for t in cache)
print("final cache size for this toy layer:", cache_bytes, "bytes")

step 1: fed 'the' | cache now holds 1 keys & values
step 2: fed 'cat' | cache now holds 2 keys & values
step 3: fed 'sat' | cache now holds 3 keys & values
step 4: fed 'on' | cache now holds 4 keys & values
step 5: fed 'mat' | cache now holds 5 keys & values

incremental (cached) == full pass: True
final cache size for this toy layer: 320 bytes


## Takeaways

- **Without a cache**, step *t* recomputes K and V for all *t* tokens (and attention over all pairs), so generating *n* tokens costs roughly O(n²) projection work in total.
- **With a cache**, each step projects only **one** new token and attends over the stored keys, so projection work is O(1) per step and attention is O(t) per step.
- **Trade-off: memory.** The cache stores `2 (K and V) x n_layers x n_heads x d_k x seq_len x batch x bytes_per_value`. For long contexts or big batches this becomes the main memory cost of LLM serving, which is why techniques like multi-query / grouped-query attention, paged attention, and cache quantization exist.
- **Two phases in inference:** *prefill* processes the whole prompt at once (cache empty, `new_len = prompt length`), then *decode* feeds one token at a time using the cache. The same function handles both, because the mask slice `[past_len:total_len, :total_len]` adapts to any `new_len`.
- **Each layer has its own cache.** This notebook shows a single attention layer; a real model keeps one `(k, v)` pair per layer.